# MERIT-Net 10-seed evaluation on Colab
Runs the remaining closed-loop NS-3 simulations **on every CPU core** of the runtime, writing results to Google Drive so a disconnect loses nothing.

**Before starting:** set `REPO_URL` in step 2 to this repository's GitHub address.

**Runtime choice:** *Runtime > Change runtime type*. The work is CPU-bound, so pick the runtime with the **most CPU cores** your plan offers. Run cell 1 after switching to see the core count. TPU runtimes usually have the most CPU cores; the free CPU/T4 runtime has about 2, which is no faster than before.

This notebook runs seeds **20 → 13** (descending). The other machine runs 13 → 20, so the two never duplicate work for long.

In [ ]:
# 1. How many CPU cores does this runtime have?
!nproc
!free -g | head -2

In [ ]:
# 2. Mount Google Drive (results are saved there) and clone the repository
REPO_URL = 'https://github.com/YOUR-GITHUB-USERNAME/MERIT-Net.git'   # <- set this
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/meritnet_colab && git clone -q $REPO_URL /content/meritnet_colab && ls /content/meritnet_colab

In [ ]:
# 3. Install the exact versions used to produce the paper's results
#    (same NS-3 build = identical simulations; same scikit-learn = the saved kNN/RF models load)
!pip install -q ns3==3.44.post0 scikit-learn==1.9.1 joblib scipy
import sklearn, torch; print('sklearn', sklearn.__version__, '| torch', torch.__version__)

In [ ]:
# 4. Build the NS-3 scenario and run a 20-second smoke test
%cd /content/meritnet_colab/ns3sim
!chmod +x build_portable.sh && ./build_portable.sh meritnet-scenario.cc meritnet-scenario
!./meritnet-scenario --nNodes=50 --speed=5 --sim=20 --seed=1 --out=/tmp/smoke.bin && echo SMOKE TEST OK

In [ ]:
# 5. Results folder on Drive (seeded with the results finished so far, only on first use)
import os, shutil, glob
RES = '/content/drive/MyDrive/meritnet_results'
os.makedirs(RES, exist_ok=True)
for f in glob.glob('/content/meritnet_colab/ns3sim/results/*.jsonl'):
    dst = os.path.join(RES, os.path.basename(f))
    if not os.path.exists(dst): shutil.copy(f, dst)
!wc -l {RES}/*.jsonl

In [ ]:
# 6. RUN. Uses all cores, seeds 20 -> 13. Re-run this cell after any disconnect: it resumes.
#    Keep the browser tab open; Colab may disconnect idle sessions.
%cd /content/meritnet_colab/ns3sim
!RESULTS_DIR=/content/drive/MyDrive/meritnet_results SEED_ORDER=desc python3 -u run_10seeds.py 2>&1 | grep --line-buffered -v Warning

In [ ]:
# 7. Progress: completed runs per seed (each seed needs 270 = 9 detectors x 30 scenarios)
import json, collections
c = collections.Counter()
for f in glob.glob(RES + '/*.jsonl'):
    for l in open(f): c[json.loads(l)['seed']] += 1
for s in sorted(c): print('seed', s, c[s])

## When it finishes
Copy `ns3_10seed_main.jsonl` and `ns3_ablation_180runs.jsonl` from the `meritnet_results` folder on Drive into `ns3sim/results/`, then run `python3 analysis/stats.py`, `python3 analysis/make_figs.py`. Duplicate runs are removed when results are merged, and the tables, confidence intervals and figures regenerate.